In [ ]:
import os
import requests
from dotenv import load_dotenv
import time


load_dotenv("test.env")
api_key_TMDB = os.getenv("TMDB_API_KEY")


if not api_key_TMDB:
    print("API KEY not found check env file")
    exit()
else:
    print("KEY loaded")


# Frage 10
Build Api accesslink + get data



In [2]:
# 2. Build link
base_url = "https://api.themoviedb.org/3"
endpoint = "/discover/movie"  #Get all result for "" fitting criteria
url = base_url + endpoint




In [ ]:
# Unsere Liste, in die wir nach und nach alle Filme reinwerfen
all_top_movies = []


# 1page=20 movies
for actual_page in range(1, 3):
    
    # Filter 
    parameter = {
        "api_key": api_key_TMDB,
        "primary_release_year": 2010,
        "language": "de-DE",
        "sort_by": "popularity.desc",
        "page": actual_page  # Count up for each
    }
    
    # Send api request
    response = requests.get(url, params=parameter)
    
    if response.status_code == 200:
        daten = response.json()
        filme_auf_seite = daten["results"]
        
        
        # For each film per page
        for movie in filme_auf_seite:
            tmdb_id = movie.get("id")
            imdb_id = "Nicht gefunden"
            
            # request link to get imdb id
            detail_url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
            detail_params = {"api_key": api_key_TMDB}
            
            # send request
            try:
                detail_response = requests.get(detail_url, params=detail_params)
                if detail_response.status_code == 200:
                    detail_daten = detail_response.json()
                    imdb_id = detail_daten.get("imdb_id", "Nicht gefunden")
            except:
                pass 
                
            
            movie["imdb_id"] = imdb_id
            
            # add movie to result
            all_top_movies.append(movie)
            
            # Spam protection
            time.sleep(0.1)
      
            
        print(f"Loaded page {actual_page}.")
            
    else:
        print(f"Error on page{actual_page}")
        break 
        
    # Spam protection for pages
    time.sleep(0.3) 

# Finished
print(f"loaded {len(all_top_movies)} movies.\n")

print("This is the last entry")
print("-" * 40)
last_movie = all_top_movies[-1]
print(f"Titel: {last_movie.get('title')} | IMDb-ID: {last_movie.get('imdb_id')}")
print("-" * 40)
    

In [ ]:
import csv

# file name
filename = "movies_from_year-2010.csv"

print("write data in CSV")

# give writing perm and add utf-8 for "Sonderzeichen"
with open(filename, mode='w', newline='', encoding='utf-8') as datei:
    
    # define lines
    columns = ['id', 'title', 'rating', 'release_date', 'original_language', 'IMDb_ID']
    writer = csv.DictWriter(datei, fieldnames=columns)

    # write headline
    writer.writeheader()

    # add each row
    for index, film in enumerate(all_top_movies, start=1):
        writer.writerow({
            'id': index,
            'title': film.get('title'),
            'rating': film.get('vote_average'),
            'release_date': film.get('release_date'),
            'original_language': film.get('original_language'),
            'IMDb_ID': film.get('imdb_id', 'Nicht gefunden') 
        })

print(f"Finished writing '{filename}'")